# Video Inference For Ad Surface Detection

Простой ноутбук для проверки обученной YOLO-модели на видео `input_video/test_video.mp4`. Результаты сохраняются в `output_video/`.

## Config

По умолчанию берём последний эксперимент `yolo11m_img1280_ep75`. Если его весов нет, ячейка автоматически возьмёт самый свежий `best.pt` из `runs/`.

In [1]:
from pathlib import Path

import cv2
import pandas as pd
from IPython.display import Video, display
from ultralytics import YOLO

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists()),
    Path.cwd(),
)

VIDEO_PATH = PROJECT_ROOT / "input_video" / "test_video_2.mp4"
OUTPUT_DIR = PROJECT_ROOT / "output_video"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_WEIGHTS = PROJECT_ROOT / "runs" / "detect" / "runs_ad_surface_experiments" / "yolo11m_img1280_ep75" / "weights" / "best.pt"
if not MODEL_WEIGHTS.exists():
    MODEL_WEIGHTS = sorted(
        (PROJECT_ROOT / "runs").rglob("weights/best.pt"),
        key=lambda path: path.stat().st_mtime,
    )[-1]

CONF = 0.70
IMGSZ = 1280
VID_STRIDE = 1
TRACKER = "bytetrack.yaml"
RUN_NAME = "test_video_track_conf70_bytetrack"

assert VIDEO_PATH.exists(), f"Video not found: {VIDEO_PATH}"
assert MODEL_WEIGHTS.exists(), f"Weights not found: {MODEL_WEIGHTS}"

print("Video:", VIDEO_PATH.resolve())
print("Output dir:", OUTPUT_DIR.resolve())
print("Weights:", MODEL_WEIGHTS.resolve())
print("Tracker:", TRACKER)


Video: /home/shiawase/ic8_ai/ad_detection/test_video_2.mp4
Output dir: /home/shiawase/ic8_ai/ad_detection/output_video
Weights: /home/shiawase/ic8_ai/ad_detection/runs/detect/runs_ad_surface_experiments/yolo11m_img1280_ep75/weights/best.pt
Tracker: bytetrack.yaml


## Video Metadata

Смотрим длительность, FPS и размер кадра до запуска инференса.

In [2]:
cap = cv2.VideoCapture(str(VIDEO_PATH))

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration_sec = frame_count / fps if fps else None

cap.release()

video_info = pd.DataFrame([{
    "path": str(VIDEO_PATH),
    "fps": fps,
    "frames": frame_count,
    "duration_sec": duration_sec,
    "width": width,
    "height": height,
    "vid_stride": VID_STRIDE,
}])

video_info

,path,fps,frames,duration_sec,width,height,vid_stride
0,test_video_2.mp4,29.992732,11182,372.823656,720,1280,1


## Run Tracking

Тест с ByteTrack и высоким порогом confidence: в итоговом видео будут показаны только bbox с уверенностью от 70%. Результат сохраняется отдельно в `output_video/test_video_track_conf70_bytetrack/`.

In [3]:
model = YOLO(str(MODEL_WEIGHTS))

results_stream = model.track(
    source=str(VIDEO_PATH),
    conf=CONF,
    imgsz=IMGSZ,
    vid_stride=VID_STRIDE,
    tracker=TRACKER,
    save=True,
    save_txt=False,
    save_conf=False,
    stream=True,
    project=str(OUTPUT_DIR.resolve()),
    name=RUN_NAME,
    exist_ok=True,
)

rows = []
processed_frames = 0
save_dir = None

for result in results_stream:
    processed_frames += 1
    save_dir = Path(result.save_dir)

    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        continue

    xyxy = boxes.xyxy.cpu().numpy()
    confs = boxes.conf.cpu().numpy()
    classes = boxes.cls.cpu().numpy().astype(int)
    track_ids = boxes.id.cpu().numpy().astype(int) if boxes.id is not None else [None] * len(boxes)

    for box_idx, (coords, conf, class_id, track_id) in enumerate(zip(xyxy, confs, classes, track_ids), start=1):
        x1, y1, x2, y2 = coords
        rows.append({
            "processed_frame_idx": processed_frames,
            "box_idx": box_idx,
            "track_id": track_id,
            "class_id": class_id,
            "confidence": float(conf),
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
        })

detections_df = pd.DataFrame(rows)
detections_csv_path = OUTPUT_DIR / f"{RUN_NAME}_detections.csv"
detections_df.to_csv(detections_csv_path, index=False)

print("Processed frames:", processed_frames)
print("Detections:", len(detections_df))
print("Save dir:", save_dir)
print("Detections CSV:", detections_csv_path)

detections_df.head(20)


video 1/1 (frame 1/11182) /home/shiawase/ic8_ai/ad_detection/test_video_2.mp4: 1280x736 (no detections), 53.3ms
video 1/1 (frame 2/11182) /home/shiawase/ic8_ai/ad_detection/test_video_2.mp4: 1280x736 (no detections), 13.3ms
video 1/1 (frame 3/11182) /home/shiawase/ic8_ai/ad_detection/test_video_2.mp4: 1280x736 (no detections), 13.5ms
video 1/1 (frame 4/11182) /home/shiawase/ic8_ai/ad_detection/test_video_2.mp4: 1280x736 (no detections), 13.3ms
video 1/1 (frame 5/11182) /home/shiawase/ic8_ai/ad_detection/test_video_2.mp4: 1280x736 (no detections), 13.1ms
video 1/1 (frame 6/11182) /home/shiawase/ic8_ai/ad_detection/test_video_2.mp4: 1280x736 (no detections), 13.2ms
video 1/1 (frame 7/11182) /home/shiawase/ic8_ai/ad_detection/test_video_2.mp4: 1280x736 (no detections), 13.7ms
video 1/1 (frame 8/11182) /home/shiawase/ic8_ai/ad_detection/test_video_2.mp4: 1280x736 (no detections), 13.4ms
video 1/1 (frame 9/11182) /home/shiawase/ic8_ai/ad_detection/test_video_2.mp4: 1280x736 (no detections)

,processed_frame_idx,box_idx,track_id,class_id,confidence,x1,y1,x2,y2
0,833,1,NaN,0,0.766045,236.765137,919.014893,254.761963,986.696289
1,834,1,1.0,0,0.853159,235.211121,927.401794,253.546539,996.356079
2,835,1,1.0,0,0.823530,232.144455,936.492554,250.924881,1007.106567
3,836,1,1.0,0,0.836845,228.318054,946.703857,248.223709,1021.587769
4,837,1,1.0,0,0.746105,225.416412,958.488464,245.809341,1035.297729
5,1543,1,NaN,0,0.803721,369.082886,868.230103,420.499084,948.759644
6,1544,1,2.0,0,0.851653,368.413696,869.066528,421.561096,952.329651
7,1545,1,2.0,0,0.743454,368.380096,869.827393,422.796478,955.146057
8,1591,1,NaN,0,0.704026,368.037323,869.153076,430.399445,982.145508
9,1594,1,NaN,0,0.724671,366.772827,868.172852,431.777527,1041.934082


## Detection Summary

Сводка по кадрам: где модель нашла больше всего объектов и какие confidence у детекций.

In [4]:
if detections_df.empty:
    print("No detections found")
else:
    frame_summary_df = detections_df.groupby("processed_frame_idx").agg(
        objects_count=("box_idx", "count"),
        mean_confidence=("confidence", "mean"),
        max_confidence=("confidence", "max"),
    ).reset_index().sort_values("objects_count", ascending=False)

    display(frame_summary_df.head(20))
    display(detections_df["confidence"].describe().to_frame().T)

,processed_frame_idx,objects_count,mean_confidence,max_confidence
408,2007,2,0.853483,0.899829
409,2008,2,0.854954,0.900970
410,2009,2,0.856452,0.902192
188,1774,2,0.793610,0.885933
187,1773,2,0.830630,0.886248
411,2010,2,0.839455,0.901395
698,4281,2,0.853754,0.876114
185,1771,2,0.801976,0.880818
412,2011,2,0.814609,0.899812
671,2290,2,0.863194,0.880869


,count,mean,std,min,25%,50%,75%,max
confidence,1240.0,0.85073,0.056554,0.700182,0.81552,0.878116,0.892732,0.928677


## View Saved Tracking Video

Показываем видео, которое сохранил `model.track`.

In [5]:
saved_videos = sorted(save_dir.glob("*.mp4")) if save_dir else []

print("Saved videos:", [str(path) for path in saved_videos])

if saved_videos:
    display(Video(str(saved_videos[0]), embed=False, html_attributes="controls width=960"))

Saved videos: []


## Run Smoothed Tracking

Трекер получает детекции с низким порогом `TRACK_CONF = 0.25`, но на видео показываем только стабильные треки: начать показ при `START_CONF >= 0.70`, продолжать при `KEEP_CONF >= 0.45`, держать последний bbox до 5 пропущенных кадров.

In [6]:
TRACK_CONF = 0.25
START_CONF = 0.70
KEEP_CONF = 0.45
MAX_MISSING_FRAMES = 5
SMOOTH_RUN_NAME = "test_video_track_smooth_start70_keep45_hold5"

smooth_output_dir = OUTPUT_DIR / SMOOTH_RUN_NAME
smooth_output_dir.mkdir(parents=True, exist_ok=True)

smooth_video_path = smooth_output_dir / f"{VIDEO_PATH.stem}_{SMOOTH_RUN_NAME}.mp4"
smooth_csv_path = OUTPUT_DIR / f"{SMOOTH_RUN_NAME}_detections.csv"

print("Smooth output video:", smooth_video_path)
print("Smooth detections CSV:", smooth_csv_path)

Smooth output video: output_video/test_video_track_smooth_start70_keep45_hold5/test_video_2_test_video_track_smooth_start70_keep45_hold5.mp4
Smooth detections CSV: output_video/test_video_track_smooth_start70_keep45_hold5_detections.csv


In [7]:
def draw_label(frame, text, x1, y1, color):
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    thickness = 2
    padding = 4

    text_size, baseline = cv2.getTextSize(text, font, font_scale, thickness)
    text_w, text_h = text_size

    bg_x1 = x1
    bg_y1 = max(0, y1 - text_h - baseline - padding * 2)
    bg_x2 = min(frame.shape[1] - 1, x1 + text_w + padding * 2)
    bg_y2 = y1

    cv2.rectangle(frame, (bg_x1, bg_y1), (bg_x2, bg_y2), color, -1)
    cv2.putText(
        frame,
        text,
        (bg_x1 + padding, bg_y2 - baseline - padding),
        font,
        font_scale,
        (255, 255, 255),
        thickness,
        cv2.LINE_AA,
    )


def draw_track(frame, track_id, track_state):
    x1, y1, x2, y2 = [int(round(value)) for value in track_state["bbox"]]
    x1 = max(0, min(frame.shape[1] - 1, x1))
    x2 = max(0, min(frame.shape[1] - 1, x2))
    y1 = max(0, min(frame.shape[0] - 1, y1))
    y2 = max(0, min(frame.shape[0] - 1, y2))

    color = (255, 0, 0)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)

    suffix = "" if track_state["missing"] == 0 else f" hold:{track_state['missing']}"
    label = f"id:{track_id} ad_surface {track_state['confidence']:.2f}{suffix}"
    draw_label(frame, label, x1, y1, color)

In [8]:
smooth_model = YOLO(str(MODEL_WEIGHTS))
cap = cv2.VideoCapture(str(VIDEO_PATH))

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

writer = cv2.VideoWriter(
    str(smooth_video_path),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
)

active_tracks = {}
rows = []
frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame_idx += 1

    result = smooth_model.track(
        source=frame,
        conf=TRACK_CONF,
        imgsz=IMGSZ,
        tracker=TRACKER,
        persist=True,
        verbose=False,
    )[0]

    seen_track_ids = set()

    if result.boxes is not None and len(result.boxes) > 0 and result.boxes.id is not None:
        xyxy = result.boxes.xyxy.cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)
        track_ids = result.boxes.id.cpu().numpy().astype(int)

        detections_by_track = {}
        for bbox, conf, class_id, track_id in zip(xyxy, confs, classes, track_ids):
            current = detections_by_track.get(track_id)
            if current is None or conf > current["confidence"]:
                detections_by_track[track_id] = {
                    "bbox": bbox,
                    "confidence": float(conf),
                    "class_id": int(class_id),
                }

        for track_id, detection in detections_by_track.items():
            seen_track_ids.add(track_id)
            is_active = track_id in active_tracks
            conf = detection["confidence"]

            if not is_active and conf >= START_CONF:
                active_tracks[track_id] = {
                    "bbox": detection["bbox"],
                    "confidence": conf,
                    "class_id": detection["class_id"],
                    "missing": 0,
                }
            elif is_active:
                if conf >= KEEP_CONF:
                    active_tracks[track_id].update({
                        "bbox": detection["bbox"],
                        "confidence": conf,
                        "class_id": detection["class_id"],
                        "missing": 0,
                    })
                else:
                    active_tracks[track_id]["missing"] += 1

    for track_id in list(active_tracks):
        if track_id not in seen_track_ids:
            active_tracks[track_id]["missing"] += 1

        if active_tracks[track_id]["missing"] > MAX_MISSING_FRAMES:
            del active_tracks[track_id]

    for track_id, track_state in active_tracks.items():
        draw_track(frame, track_id, track_state)
        x1, y1, x2, y2 = [float(value) for value in track_state["bbox"]]
        rows.append({
            "frame_idx": frame_idx,
            "track_id": track_id,
            "class_id": track_state["class_id"],
            "confidence": track_state["confidence"],
            "missing": track_state["missing"],
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
        })

    writer.write(frame)

    if frame_idx % 500 == 0:
        print(f"Processed {frame_idx}/{frame_count} frames")

cap.release()
writer.release()

detections_smoothed_df = pd.DataFrame(rows)
detections_smoothed_df.to_csv(smooth_csv_path, index=False)

print("Done")
print("Frames:", frame_idx)
print("Smoothed detections rows:", len(detections_smoothed_df))
print("Video:", smooth_video_path)
print("CSV:", smooth_csv_path)

detections_smoothed_df.head(20)

Processed 500/11182 frames
Processed 1000/11182 frames
Processed 1500/11182 frames
Processed 2000/11182 frames
Processed 2500/11182 frames
Processed 3000/11182 frames
Processed 3500/11182 frames
Processed 4000/11182 frames
Processed 4500/11182 frames
Processed 5000/11182 frames
Processed 5500/11182 frames
Processed 6000/11182 frames
Processed 6500/11182 frames
Processed 7000/11182 frames
Processed 7500/11182 frames
Processed 8000/11182 frames
Processed 8500/11182 frames
Processed 9000/11182 frames
Processed 9500/11182 frames
Processed 10000/11182 frames
Processed 10500/11182 frames
Processed 11000/11182 frames
Done
Frames: 11182
Smoothed detections rows: 1787
Video: output_video/test_video_track_smooth_start70_keep45_hold5/test_video_2_test_video_track_smooth_start70_keep45_hold5.mp4
CSV: output_video/test_video_track_smooth_start70_keep45_hold5_detections.csv


,frame_idx,track_id,class_id,confidence,missing,x1,y1,x2,y2
0,833,27,0,0.766045,0,237.494354,917.661682,255.431046,985.384216
1,834,27,0,0.853159,0,235.109863,928.013489,253.462036,997.285645
2,835,27,0,0.823530,0,231.880035,938.105225,250.712036,1009.125610
3,836,27,0,0.836845,0,228.347610,947.750854,248.163345,1022.560181
4,837,27,0,0.746105,0,225.528763,958.949768,245.806747,1035.666504
5,838,27,0,0.649739,0,222.558472,970.467712,241.553070,1041.299927
6,839,27,0,0.554285,0,217.758392,983.221924,238.965561,1062.731079
7,840,27,0,0.554285,1,217.758392,983.221924,238.965561,1062.731079
8,841,27,0,0.554285,2,217.758392,983.221924,238.965561,1062.731079
9,842,27,0,0.554285,3,217.758392,983.221924,238.965561,1062.731079


In [9]:
if smooth_video_path.exists():
    display(Video(str(smooth_video_path), embed=False, html_attributes="controls width=960"))